## Preprocessing: Dataset B (Aljolfey et al., 2022 benchmark)

This notebook outlines the conversion of the Aljolfey et al. (2022) benchmark in the same per-sample JSON schema and manifest format that `1-scrape.py` produces for Dataset A, so both datasets can be run through the same downstream feature extraction and preprocessing pipeline unmodified.

### Role:
1. Load the `id, url, typ` benchmark manifest and map typ to this study's label convention
2. Match each id to its archieved raw HTML capture on disk, parse it, and extrac the same fields for `Dataset A`. Fields: `http_status`, `ssl_info`, and `whois_info` are explicitly set to `None` since these are not fetched live by this study
3. Flag archived captures that are block pages rather than real site content, logging them separately instead of silently treating them as valid samples
4. Write one JSON record per sample plus a manifest.csv, and report class balance and discard counts.

In [1]:
# library imports
import csv
import hashlib
import json
import logging
import os
import sys
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from datetime import datetime, timezone
from urllib.parse import urlparse
from bs4 import BeautifulSoup

In [2]:
# directory mapping
MANIFEST_CSV = "DB/D1/ID__URLs_labell.csv"
HTML_DIR = "DB/D1/sourc_Data(D1)/"
RAW_DIR_B = "DB/preprocessed_D1"
LOG_PATH = "logs/preprocess_aljolfey.log"

manifest_out_path = os.path.join(RAW_DIR_B, "manifest.csv")
flagged_path = os.path.join(RAW_DIR_B, "flagged_pages.csv")
missing_path = os.path.join(RAW_DIR_B, "missing_html_files.csv")
malformed_path = os.path.join(RAW_DIR_B, "malformed_urls.csv")
fetch_error_path = os.path.join(RAW_DIR_B, "fetch_errors.csv")

os.makedirs(RAW_DIR_B, exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [3]:
# logging configuration
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.flush = sys.stdout.flush
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), stream_handler],
    force=True,
)
log = logging.getLogger(__name__)

In [4]:
# label mapping
LABEL_MAP = {"+1": 1, "-1": 0}

MANIFEST_FIELDS = [
    "sample_id", "source", "original_id", "seed_url", "final_url",
    "hostname", "http_status", "scraped_at", "n_inline_scripts",
    "n_external_scripts", "flagged_block_page", "label",
]

BLOCK_PAGE_MARKERS = [
    "attention required! | cloudflare",
    "cf-error-details",
    "cf-wrapper",
    "just a moment...",
    "checking your browser before accessing",
    "captcha-bypass",
    "please enable cookies",
    "ddos protection by",
]

# Some errors are detected by pattern rather than an enumerated phrase list
# since wording valids and a fixed list can never cover every variant
VOCAB = (
    r"(?:error|not\s*found|forbidden|unauthorized|bad\s*request|unavailable|"
    r"denied|timeout|timed\s*out|gateway|internal\s*server(?:\s*error)?|"
    r"too\s*many\s*requests|method\s*not\s*allowed|moved\s*permanently|"
    r"service\s*unavailable)"
)

SEP = r"\s*[:\-]?\s*"
HTTP_ERROR_ADJACENT_PATTERN = re.compile(
    rf"(?:\b(?:http\s*)?{VOCAB}{SEP}([1-5]\d{{2}})\b)"
    rf"|(?:\b([1-5]\d{{2}}){SEP}{VOCAB}\b)",
    re.IGNORECASE,
)

CONNECTION_ERROR_MARKERS = [
    "urlopen error", "urllib.error", "connection refused", "connection timed out",
    "err_connection", "err_name_not_resolved", "err_empty_response",
    "dns_probe_finished", "name or service not known",
    "temporary failure in name resolution", "no address associated with hostname",
    "this site can't be reached", "this site can not be reached",
    "ssl: certificate_verify_failed", "certificate verify failed",
]

FETCH_ERROR_LENGTH_CUTOFF = 500  # captures longer than this are treated as content

HTTP_ERROR_PHRASE_MARKERS = [
    "403 forbidden",
    "access denied",
]

HTML_TOO_SHORT_CUTOFF = 200

### Helper function

In [5]:
def normalise(url):
    """Normalise URL for hashing"""
    parsed = urlparse(url if "://" in url else f"https://{url}")
    protocol = "https" if parsed.scheme == "https" else "http"
    hostname = (parsed.hostname or "").lower()
    try:
        if parsed.port and parsed.port not in (80, 443):
            hostname = f"{hostname}:{parsed.port}"
    except ValueError:
        return None
    path = parsed.path.rstrip("/")
    return f"{protocol}://{hostname}{path}"

def sample_id_for(url):
    """sample hash scheme"""
    return hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]

def extract_scripts(soup):
    inline, external_srcs = [], []
    for tag in soup.find_all("script"):
        if tag.get("src"):
            external_srcs.append(tag["src"])
        elif tag.string:
            inline.append(tag.string)
    return inline, external_srcs

def detect_block_page(html, title_text):
    """heuristic flag for archived captures that are block pages"""
    haystack = f"{title_text} {html[:3000]}".lower()
    for marker in BLOCK_PAGE_MARKERS:
        if marker in haystack:
            return True, marker
    return False, None

def might_be_fetch_error(html):
    """
    Detects captures that are a raw fetch error string rather real HTML
    Accounts for 3 cases:
        1. no HTML markup at all
        2. a status-code-like number directly adject to generic HTTP error vocab
        3. a known connection/DNS-level failure phrase with no status code involved    
    """
    stripped = html.strip()
    if not stripped:
        return True, "empty_capture"

    has_markup = "<" in stripped and ">" in stripped
    short = len(stripped) < FETCH_ERROR_LENGTH_CUTOFF

    if not has_markup and len(stripped) < 300:
        return True, "no_html_markup"

    haystack = stripped[:3000].lower()
    for marker in HTTP_ERROR_PHRASE_MARKERS:
        if marker in haystack:
            return True, marker
        
    if len(stripped) < HTML_TOO_SHORT_CUTOFF:
        return True, "html_too_short"

    head = stripped[:FETCH_ERROR_LENGTH_CUTOFF]

    if short:
        text = re.sub(r"<[^>]+>", " ", head).strip()
        match = HTTP_ERROR_ADJACENT_PATTERN.search(text)
        if match:
            code = match.group(1) or match.group(2)
            if code and code[0] in ("4", "5"):
                if len(text) <= 60 or (match.end() - match.start()) / max(len(text), 1) > 0.5:
                    return True, f"http_error_{code}"

    for marker in CONNECTION_ERROR_MARKERS:
        if marker in head.lower():
            return True, marker

    return False, None

### Loading the Aljolfey manifest

In [6]:
entries = {}
skipped_unknown_label = 0

with open(MANIFEST_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        typ = row.get("typ")
        if typ not in LABEL_MAP:
            skipped_unknown_label += 1
            continue
        entries[row["id"]] = (row["url"].strip(), LABEL_MAP[typ])

if skipped_unknown_label:
    log.warning(f"Skipped {skipped_unknown_label} rows with unrecongnised 'typ' values.")
log.info(f"Loaded {len(entries)} labelled entries from {MANIFEST_CSV}")

2026-09-29 22:21:27,357 [INFO] Loaded 60252 labelled entries from D1/ID__URLs_labell.csv


### Converting each sample to match Dataset A schema

In [7]:
progress_every = 2000

manifest_rows = []
flagged_records = []
missing_records = []
malformed_records = []
fetch_error_records = []

for i, (original_id, (url, label)) in enumerate(entries.items(), start=1):
    html_path = os.path.join(HTML_DIR, f"{original_id}.txt")
    if not os.path.exists(html_path):
        missing_records.append({"original_id": original_id, "url": url})
        continue

    key = normalise(url)
    if key is None:
        malformed_records.append({
            "source": "D1", "url": url,
            "original_id": original_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        continue

    sid = sample_id_for(key)

    with open(html_path, "r", encoding="utf-8") as f:
        raw_html = f.read()

    is_fetch_error, fetch_error_reason = might_be_fetch_error(raw_html)
    is_fetch_error = 1 if is_fetch_error else 0
    if is_fetch_error:
        fetch_error_records.append({"original_id": original_id, "url": url, "reason": fetch_error_reason})
        continue

    try:
        soup = BeautifulSoup(raw_html, "html.parser")
    except Exception as e:
        log.warning(f"[discard] id={original_id} url={url} -> HTML parse error: {e}")
        malformed_records.append({
            "source": "D1", "url": url,
            "original_id": original_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        continue

    title_tag = soup.find("title")
    title_text = title_tag.get_text() if title_tag else ""
    flagged, reason = detect_block_page(raw_html, title_text)
    flagged = 1 if flagged else 0

    inline_scripts, external_script_srcs = extract_scripts(soup)
    hostname = urlparse(key).hostname
    scraped_url = url if url.startswith(("http://", "https://")) else f"http://{url}"
    scraped_at = datetime.now(timezone.utc).isoformat()

    record = {
        "sample_id": sid,
        "source": "D1",
        "original_id": original_id,
        "listed_at": None,
        "seed_url": scraped_url,
        "final_url": scraped_url,
        "hostname": hostname,
        "http_status": None,
        "scraped_at": scraped_at,
        "raw_html": raw_html,
        "inline_scripts": inline_scripts,
        "external_script_srcs": external_script_srcs,
        "ssl_info": None,
        "whois_info": None,
        "flagged_block_page": flagged,
        "flag_reason": reason,
        "label": label,
    }

    out_path = os.path.join(RAW_DIR_B, f"{sid}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(record, f)

    manifest_rows.append({
        "sample_id": sid, "source": "D1", "original_id": original_id,
        "seed_url": scraped_url, "final_url": scraped_url,
        "hostname": hostname, "http_status": "", "scraped_at": scraped_at,
        "n_inline_scripts": len(inline_scripts), "n_external_scripts": len(external_script_srcs),
        "flagged_block_page": flagged, "label": label,
    })

    if flagged:
        flagged_records.append({"original_id": original_id, "sample_id": sid, "url": url, "reason": reason})

    if i % progress_every == 0:
        log.info(f"Processed {i} / {len(entries)} Aljolfey ids")

log.info(
    f"****** Conversion complete: {len(manifest_rows)} / {len(entries)} converted "
    f"({len(missing_records)} missing HTML files), {len(malformed_records)} malformed URLs, "
    f"{len(fetch_error_records)} raw fetch-error captures included, "
    f"{len(flagged_records)} flagged as likely block pages) ******"
)

2026-09-29 22:21:33,415 [INFO] Processed 2000 / 60252 Aljolfey ids
2026-09-29 22:21:40,703 [INFO] Processed 6000 / 60252 Aljolfey ids
2026-09-29 22:21:51,675 [INFO] Processed 12000 / 60252 Aljolfey ids
2026-09-29 22:21:54,445 [INFO] Processed 14000 / 60252 Aljolfey ids
2026-09-29 22:21:58,277 [INFO] Processed 16000 / 60252 Aljolfey ids
2026-09-29 22:22:01,942 [INFO] Processed 18000 / 60252 Aljolfey ids
2026-09-29 22:22:18,179 [INFO] Processed 22000 / 60252 Aljolfey ids
2026-09-29 22:22:31,577 [INFO] Processed 24000 / 60252 Aljolfey ids
2026-09-29 22:22:41,700 [INFO] Processed 26000 / 60252 Aljolfey ids
2026-09-29 22:23:06,471 [INFO] Processed 30000 / 60252 Aljolfey ids
2026-09-29 22:23:10,764 [INFO] Processed 32000 / 60252 Aljolfey ids
2026-09-29 22:23:13,972 [INFO] Processed 34000 / 60252 Aljolfey ids
2026-09-29 22:23:28,417 [INFO] Processed 38000 / 60252 Aljolfey ids
2026-09-29 22:23:40,606 [INFO] Processed 40000 / 60252 Aljolfey ids
2026-09-29 22:23:54,167 [INFO] Processed 42000 / 6

### Building the manifest dataframe and reporting class balance

In [8]:
df = pd.DataFrame(manifest_rows)
df.head()

,sample_id,source,original_id,seed_url,final_url,hostname,http_status,scraped_at,n_inline_scripts,n_external_scripts,flagged_block_page,label
0,ea38554a67b26d19,D1,5535,http://10bestbingorooms.com,http://10bestbingorooms.com,10bestbingorooms.com,,2026-09-29T20:21:29.880914+00:00,0,11,0,1
1,25cf7613b64df038,D1,5704,http://10stepstostartingyouronlinebusiness.com,http://10stepstostartingyouronlinebusiness.com,10stepstostartingyouronlinebusiness.com,,2026-09-29T20:21:29.884334+00:00,1,0,0,1
2,9bc4ed0ffbe672e3,D1,5705,http://10surdix.com,http://10surdix.com,10surdix.com,,2026-09-29T20:21:29.897388+00:00,39,124,0,1
3,12a8b89c58b8b877,D1,6123,http://116-region.ru/wp-includes/capitalone360...,http://116-region.ru/wp-includes/capitalone360...,116-region.ru,,2026-09-29T20:21:29.902154+00:00,12,13,0,1
4,3be00734fe412750,D1,6124,http://116-region.ru/wp-includes/capitalone360...,http://116-region.ru/wp-includes/capitalone360...,116-region.ru,,2026-09-29T20:21:29.906015+00:00,12,13,0,1


In [9]:
def report_balance(df, stage_label):
    counts = df["label"].value_counts()
    log.info(f"Class label [{stage_label}]: " + ", ".join(f"{k}={v}" for k, v in counts.items()))

report_balance(df, "Aljolfey (2022) Preprocessed")
log.info(
    f"[data quality] flagged block pages={df["flagged_block_page"].sum()} "
    f"of {len(df)} converted samples"
)

2026-09-29 22:28:31,254 [INFO] Class label [Aljolfey (2022) Preprocessed]: 0=29077, 1=22053
2026-09-29 22:28:31,256 [INFO] [data quality] flagged block pages=1460 of 51130 converted samples


### Writing manifest and discard logs

In [10]:
write_header = not os.path.exists(manifest_out_path)
with open(manifest_out_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
    if write_header:
        writer.writeheader()
    writer.writerows(manifest_rows)
log.info(f"Wrote manifest -> {manifest_out_path} ({len(manifest_rows)} rows)")

if flagged_records:
    write_header = not os.path.exists(flagged_path)
    with open(flagged_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_id", "sample_id", "url", "reason"])
        if write_header:
            writer.writeheader()
        writer.writerows(flagged_records)
    log.info(f"Wrote flagged block page log -> {flagged_path} ({len(flagged_records)} rows)")

if missing_records:
    write_header = not os.path.exists(missing_path)
    with open(missing_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_id", "url"])
        if write_header:
            writer.writeheader()
        writer.writerows(missing_records)
    log.info(f"Wrote missing file log -> {missing_path} ({len(missing_records)} rows)")

if malformed_records:
    write_header = not os.path.exists(missing_path)
    with open(malformed_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "url", "original_id", "timestamp"])
        if write_header:
            writer.writeheader()
        writer.writerows(malformed_records)
    log.info(f"Wrote malformed url log -> {malformed_path} ({len(malformed_records)} rows)")

if fetch_error_records:
    write_header = not os.path.exists(fetch_error_path)
    with open(fetch_error_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_id", "url", "reason"])
        if write_header:
            writer.writeheader()
        writer.writerows(fetch_error_records)
    log.info(f"Wrote malformed url log -> {fetch_error_path} ({len(fetch_error_records)} rows)")

2026-09-29 22:28:34,496 [INFO] Wrote manifest -> preprocessed_D1/manifest.csv (51130 rows)
2026-09-29 22:28:34,500 [INFO] Wrote flagged block page log -> preprocessed_D1/flagged_pages.csv (1460 rows)
2026-09-29 22:28:34,501 [INFO] Wrote missing file log -> preprocessed_D1/missing_html_files.csv (6 rows)
2026-09-29 22:28:34,513 [INFO] Wrote malformed url log -> preprocessed_D1/fetch_errors.csv (9116 rows)


### Deduplication

In [11]:
df = pd.read_csv(manifest_out_path)
df = df.drop(columns=["http_status"])
df.head()

,sample_id,source,original_id,seed_url,final_url,hostname,scraped_at,n_inline_scripts,n_external_scripts,flagged_block_page,label
0,ea38554a67b26d19,D1,5535,http://10bestbingorooms.com,http://10bestbingorooms.com,10bestbingorooms.com,2026-09-29T20:21:29.880914+00:00,0,11,0,1
1,25cf7613b64df038,D1,5704,http://10stepstostartingyouronlinebusiness.com,http://10stepstostartingyouronlinebusiness.com,10stepstostartingyouronlinebusiness.com,2026-09-29T20:21:29.884334+00:00,1,0,0,1
2,9bc4ed0ffbe672e3,D1,5705,http://10surdix.com,http://10surdix.com,10surdix.com,2026-09-29T20:21:29.897388+00:00,39,124,0,1
3,12a8b89c58b8b877,D1,6123,http://116-region.ru/wp-includes/capitalone360...,http://116-region.ru/wp-includes/capitalone360...,116-region.ru,2026-09-29T20:21:29.902154+00:00,12,13,0,1
4,3be00734fe412750,D1,6124,http://116-region.ru/wp-includes/capitalone360...,http://116-region.ru/wp-includes/capitalone360...,116-region.ru,2026-09-29T20:21:29.906015+00:00,12,13,0,1


In [12]:
dedup_df = df.drop_duplicates(subset=["label", "hostname"], keep="first").reset_index(drop=True)
log.info(
    f"Before deduplication: {len(df)}; "
    f"After deduplication: {len(dedup_df)}"
)

2026-09-29 22:28:43,604 [INFO] Before deduplication: 51130; After deduplication: 27064


In [13]:
label_counts = dedup_df["label"].value_counts()
log.info(
    f"[reporting class balance] \n{label_counts}"
)

def report_balance(df, stage_label):
    counts = df["label"].value_counts()
    log.info(f"Class label [{stage_label}]: " + ", ".join(f"{k}={v}" for k, v in counts.items()))

2026-09-29 22:28:57,427 [INFO] [reporting class balance] 
label
0    16976
1    10088
Name: count, dtype: int64


### Dataset splits

In [14]:
train_df, test_df = train_test_split(dedup_df, test_size=0.2, random_state=42,
                                     stratify=dedup_df["label"])
report_balance(train_df, "train split")
report_balance(test_df, "test split")

2026-09-29 22:29:00,984 [INFO] Class label [train split]: 0=13581, 1=8070
2026-09-29 22:29:00,985 [INFO] Class label [test split]: 0=3395, 1=2018


In [15]:
# save clean dataset as csv
clean_manifest_path = os.path.join(RAW_DIR_B, "clean_manifest.csv")

train_df = train_df.assign(split="train")
test_df = test_df.assign(split="test")
full_df = pd.concat([train_df, test_df]).reset_index(drop=True)

full_df.to_csv(clean_manifest_path, index=False)
log.info(f"Wrote dataset manifest -> {manifest_out_path} {len(full_df)} rows")

# write train-test ids
for name, part in [("train", train_df), ("test", test_df)]:
    ids_path = os.path.join(RAW_DIR_B, f"{name}_ids.txt")
    with open(ids_path, "w") as f:
        f.write("\n".join(part["sample_id"].tolist()))
    log.info(f"Wrote {name} split IDs -> ids_path ({len(part)} ids)")

2026-09-29 22:29:04,787 [INFO] Wrote dataset manifest -> preprocessed_D1/manifest.csv 27064 rows
2026-09-29 22:29:04,790 [INFO] Wrote train split IDs -> ids_path (21651 ids)
2026-09-29 22:29:04,791 [INFO] Wrote test split IDs -> ids_path (5413 ids)
